# 02 · Language ID and topic routing
Two small, fast models run before every answer:
1. **Language ID** (english / tagalog / taglish) so the bot replies in the customer's register.
2. **Topic classifier** (18 topics) for guardrails, the claim hand-off and analytics.

All numbers are on the held-out test items (57 questions).

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src import config
pd.set_option("display.max_colwidth", 120)
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
qa = pd.read_csv(config.QA_FILE); clf = pd.read_csv(config.CLASSIFIER_FILE)
tr, te = qa[qa.split == "train"], qa[qa.split == "test"]

## 1. Language ID: rules vs. ML
The rule detector labels each token English (policy vocabulary), Tagalog (function words / affixes) or unknown. The ML detector adds character 1-4-grams + those rule features → logistic regression.

In [ ]:
from src.language import RuleLanguageDetector, MLLanguageDetector
rule = RuleLanguageDetector()
ml = MLLanguageDetector().fit(tr.input, tr.language)
for name, m in [("rule", rule), ("ml", ml)]:
    print(name, round(accuracy_score(te.language, m.predict(te.input)), 3))
ConfusionMatrixDisplay.from_predictions(te.language, ml.predict(te.input), cmap="Blues");

In [ ]:
# token-level view of a Taglish question
rule.token_labels("Covered ba kung na-delay yung flight ko ng 9 hours?")

## 2. Topic classifier
Word TF-IDF on glossary-expanded text + char 2-5-grams → logistic regression. **Weak supervision:** each policy chunk is labelled with the topic of its section heading, adding ~225 English examples for free.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from src.ingest import load_chunks
from src.topic_classifier import build_topic_model, weak_labels_from_policy

ctr = clf[clf.split == "train"]
cte = clf[(clf.split == "test") & (clf.source == "question")]
weak = weak_labels_from_policy(load_chunks())
models = {"majority": (DummyClassifier(), ctr),
          "naive_bayes": (make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), MultinomialNB(alpha=0.3)), ctr),
          "logreg": (build_topic_model(), ctr),
          "logreg + weak labels": (build_topic_model(), pd.concat([ctr[["text", "topic"]], weak]))}
for name, (m, data) in models.items():
    m.fit(data.text, data.topic)
    print(f"{name:22s} acc {accuracy_score(cte.topic, m.predict(cte.text)):.3f}")

In [ ]:
best = models["logreg + weak labels"][0]
print(classification_report(cte.topic, best.predict(cte.text), zero_division=0))

**Reading the result.** Test items are questions the model has never seen in any language, with only ~4 training items per topic, so ~46% across 18 classes is a hard setting (majority baseline 11%). In the app the topic is advisory: retrieval and Claude do not depend on it, so a wrong topic does not produce a wrong answer.